In [ ]:
import json
import pprint
import os
import pandas as pd

from google.colab import files

In [ ]:
uploaded = files.upload()

In [ ]:
!unzip -o -q /content/tdnet.zip

### コーディングでのお願い

最後のセルに、以下の情報を出力するようにして下さい：
```
新株予約権の新規発行がある開示情報の件数: XXX
新株予約権の新規発行数の合計: XXX
```


以下に、問題に対する回答をコーディングして下さい。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 正解CSV（newshare_table_utf8.csv）の列を確認
newshare_table = pd.read_csv("")
# newshare_table.columns

## 全JSONを一覧で読む（1件→44件に拡張）

In [ ]:
import os, glob, json
import pandas as pd

# 1) /content/tdnet 配下の JSON を全部列挙
json_paths = sorted(glob.glob("/content/tdnet/*.json"))

In [ ]:
# 2) JSON を全件読み込んで、1行=1開示の表（DataFrame）にする
rows = []
for p in json_paths:
    with open(p, "r", encoding="utf-8") as f:
        d = json.load(f)
    d["_json_path"] = p
    # PDFの実体パスも付けておく（後でPDFを開くときに使う）
    d["_pdf_path"] = os.path.join("/content/tdnet", d.get("pdf_file", "")) if d.get("pdf_file") else None
    rows.append(d)

tdnet_df = pd.DataFrame(rows)

# tdnet_df.shape

In [ ]:
# 4) PDFパスが実在するか確認（欠けがあると後で詰む）
tdnet_df["pdf_exists"] = tdnet_df["_pdf_path"].apply(lambda x: os.path.exists(x) if isinstance(x,str) else False)
# tdnet_df["pdf_exists"].value_counts()

In [ ]:
# 5) PDFが見つからない行があれば表示
# tdnet_df.loc[~tdnet_df["pdf_exists"], ["date","time","code","company","title","pdf_file","_pdf_path"]]

## PDF本文をテキスト化する

In [ ]:
!pip -q install pdfplumber

In [ ]:
import pdfplumber
import re
import pandas as pd

def extract_pdf_text(pdf_path: str, max_pages: int | None = None) -> str:
    """
    PDFからテキストを抽出して返す。
    失敗したら空文字を返す（後続で判定できるようにする）。
    """
    try:
        texts = []
        with pdfplumber.open(pdf_path) as pdf:
            pages = pdf.pages if max_pages is None else pdf.pages[:max_pages]
            for page in pages:
                t = page.extract_text() or ""
                texts.append(t)
        return "\n".join(texts).strip()
    except Exception as e:
        return ""

def normalize_text(s: str) -> str:
    # 解析しやすいように最低限の正規化
    s = s.replace("\u3000", " ")  # 全角スペース
    s = re.sub(r"[ \t]+", " ", s)
    return s

## Geminiで「対象判定＋発行数抽出」を構造化して返させる

In [ ]:
!pip -q install -U google-genai python-dotenv

In [ ]:
# geminiの読込

import os, json, re
from dotenv import load_dotenv
from google import genai
from google.genai import types

# .envファイルのパス。必要に応じて自身の環境のパスに変更してください。
ENV_PATH = "/path/to/.env"

load_dotenv(dotenv_path=ENV_PATH)
api_key = os.getenv("GEMINI_API_KEY")

# 外部の人がAPIキーを直接設定する場合、以下の行のコメントを外して、YOUR_API_KEYを置き換えてください。
# api_key = "YOUR_API_KEY"

if not api_key:
    raise ValueError("GEMINI_API_KEY が .env から読み込めていません。ENV_PATH と .env の中身を確認してください。")

client = genai.Client(api_key=api_key)


In [ ]:
# 新株予約権の新規発行に当たるかどうかをGeminiに判定させる

def extract_newshare_json(title: str, pdf_text: str) -> dict:
    system = (
        "あなたは情報抽出器。出力はJSONのみ。余計な文章を一切出さない。"
        "数値が特定できない場合は null。"
    )

    prompt = f"""
次の開示タイトルと本文から、以下を行ってください。

(1) この開示が「新株予約権の発行フェーズ」のどれに該当するかを判定する
(2) 「新規発行」に該当する場合のみ、『新株予約権の新規発行数』を数値で抽出する

【発行フェーズ定義（判定はこの定義に従うこと）】
フェーズ2/3 = 新規発行の意思決定・公表（＝新規発行の対象）
  - 取締役会決議/発行決議/発行に関するお知らせ
  - 条件決定/発行条件決定/発行内容確定/ストック・オプション発行内容確定
  - 第三者割当による新株予約権の発行（決議・条件確定の文脈）

フェーズ4 = 実行（払込・発行の完了）（＝対象外）
  - 払込完了/払込み完了/払込が完了/払込完了に関するお知らせ
  - 「払込完了」が本文またはタイトルに含まれる場合は必ずフェーズ4

フェーズ5 = 事後・継続イベント（＝対象外）
  - 月間行使状況/行使結果/行使状況
  - 行使価額修正（行使価格修正）/条件変更/取得/消却/譲渡/訂正/修正のみ

フェーズE = 分類不可（情報不足）（＝対象外）

【is_target の決め方（機械的に適用）】
- phase が "2or3" の場合のみ is_target=true
- phase が "4" または "5" または "E" の場合は is_target=false

【重要な優先ルール】
- タイトルまたは本文に「払込完了」系があれば、他の文言があっても必ず phase="4"、is_target=false
- タイトルまたは本文に「月間行使」「行使状況」「行使価額修正」「取得」「消却」「条件変更」「修正のみ」等が主題として含まれる場合は phase="5"、is_target=false

【数値抽出ルール（is_target=true の場合のみ）】
発行数は本文中の以下の表現から抽出する（優先順）:
1) 「発行新株予約権数」
2) 「新株予約権の総数」
3) 「新株予約権の数」
4) 「発行する新株予約権の数」
単位は "個" または "口" または "株" を返す。
数値が特定できない場合は null。

【出力JSON（キー固定。JSON以外の文章は禁止）】
{{
  "phase": "2or3" or "4" or "5" or "E",
  "is_target": true/false,
  "newshare_count": number or null,
  "unit": "個" or "口" or "株" or null,
  "evidence": "根拠箇所（短く、原文のまま）" or null
}}

開示タイトル:
{title}

本文:
{pdf_text[:20000]}
""".strip()

    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[system, prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0
        )
    )

    txt = getattr(resp, "text", None)
    if txt is None:
        txt = resp.candidates[0].content.parts[0].text

    data = json.loads(txt)

    # 数値正規化（"12,345" のような文字列対策）
    if data.get("newshare_count") is not None and isinstance(data["newshare_count"], str):
        data["newshare_count"] = int(float(data["newshare_count"].replace(",", "")))

    return data

## 正解CSVと突合して精度評価

In [ ]:
def build_candidates(tdnet_df: pd.DataFrame) -> pd.DataFrame:
    include_words = ["新株予約権", "ストック", "オプション"]
    pattern_include = "|".join(include_words)

    cand1 = tdnet_df[tdnet_df["title"].str.contains(pattern_include, na=False)].copy()
    return cand1

cand1_built = build_candidates(tdnet_df)
print(f"Shape of cand1 generated by function: {cand1_built.shape}")
print(f"First 3 rows of cand1 generated by function:\n{cand1_built.head(3)}")

In [ ]:
def build_submit_df(candidate_df: pd.DataFrame) -> pd.DataFrame:
    work_df = candidate_df.copy()

    work_df["pdf_text_raw"] = work_df["_pdf_path"].apply(lambda p: extract_pdf_text(p))
    work_df["pdf_text"] = work_df["pdf_text_raw"].apply(normalize_text)
    work_df["pdf_text_len"] = work_df["pdf_text"].str.len()

    out = []
    for _, r in work_df.iterrows():
        # LLMの入出力項目で提出に不要なフィールド（unit/evidence等）があるなら削って軽量化してよい。
        # extract_newshare_json は dict を返すので、必要なキーだけを取り出す
        extracted_data = extract_newshare_json(r["title"], r["pdf_text"])
        # 'unit' と 'evidence' は最終提出CSVに不要なため、ここで除外
        filtered_data = {k: v for k, v in extracted_data.items() if k not in ['unit', 'evidence']}
        out.append(filtered_data)

    work_df = work_df.reset_index(drop=True).join(pd.DataFrame(out))

    pred_submit = work_df[work_df["is_target"] == True].copy()

    pred_submit_out = pd.DataFrame({
        "時刻": pred_submit["time"].astype(str),
        "銘柄コード": pred_submit["code"].astype(str),
        "銘柄名": pred_submit["company"].astype(str),
        "表題": pred_submit["title"].astype(str),
        "新株予約権の新規発行数": pred_submit["newshare_count"]
    })

    key_cols = ["時刻", "銘柄コード", "銘柄名", "表題"]
    pred_submit_out = pred_submit_out.drop_duplicates(subset=key_cols, keep="first").copy()

    submit_cols = ["時刻", "銘柄コード", "銘柄名", "表題", "新株予約権の新規発行数"]
    pred_submit_out = pred_submit_out[submit_cols].copy()

    pred_submit_out["時刻"] = pred_submit_out["時刻"].astype(str)
    pred_submit_out["銘柄コード"] = pred_submit_out["銘柄コード"].astype(str)
    pred_submit_out["銘柄名"] = pred_submit_out["銘柄名"].astype(str)
    pred_submit_out["表題"] = pred_submit_out["表題"].astype(str)
    pred_submit_out["新株予約権の新規発行数"] = pd.to_numeric(pred_submit_out["新株予約権の新規発行数"], errors="coerce")

    return pred_submit_out

# cand1 の生成
cand1 = build_candidates(tdnet_df)

# pred_submit_out の生成とCSV保存
pred_submit_out = build_submit_df(cand1)
pred_submit_out.to_csv("newshare_table_pred.csv", index=False, encoding="utf-8-sig")

# 最終提出DFから件数と合計を計算
disclosure_count_final = len(pred_submit_out)
newshare_total_final = pred_submit_out["新株予約権の新規発行数"].sum()

In [ ]:
# =========================
# 1) 正解データ（Ground Truth）と予測データ（Predicted）の準備
# =========================

# 正解データ（人手作成のCSVなど）
df_gt = newshare_table.copy()

# 予測データ（Gemini等で抽出した提出用データ）
df_pred = pred_submit_out.copy()


# =========================
# 2) 結合・比較のためのキー列と値列を定義
# =========================

# 開示を一意に特定するためのキー（結合に使う列）
key_cols = ["時刻", "銘柄コード", "銘柄名", "表題"]

# 比較対象となる値（新株予約権の新規発行数）
value_col = "新株予約権の新規発行数"


# =========================
# 3) 結合が壊れないように、キー列の型を統一（文字列化）
# =========================

# 片方が数値・片方が文字列だと merge で一致しなくなるため、全て str に揃える
for col in key_cols:
    df_gt[col] = df_gt[col].astype(str)
    df_pred[col] = df_pred[col].astype(str)


# =========================
# 4) 発行数を数値に統一（数値にできない場合は NaN にする）
# =========================

# errors="coerce" により、変換できない値は NaN になる（例：空欄や文字列など）
df_gt[value_col] = pd.to_numeric(df_gt[value_col], errors="coerce")
df_pred[value_col] = pd.to_numeric(df_pred[value_col], errors="coerce")


# =========================
# 5) 正解と予測を outer join で結合し、TP/FP/FN を判定できる表を作る
# =========================

# outer で結合することで、
# - 正解にしかない行（FN候補）
# - 予測にしかない行（FP候補）
# - 両方にある行（TP候補）
# をすべて残した状態で比較できる
merged_df = pd.merge(
    df_gt,
    df_pred,
    on=key_cols,
    how='outer',
    suffixes=('_gt', '_pred'),
    indicator=True
)


# =========================
# 6) TP / FP / FN の件数を算出（開示の検出精度）
# =========================

# 正解側の値が存在するか（NaNでないか）
gt_present_mask = merged_df[value_col + '_gt'].notna()

# 予測側の値が存在するか（NaNでないか）
pred_present_mask = merged_df[value_col + '_pred'].notna()

# TP: 正解にも予測にも存在する
TP = merged_df[(gt_present_mask) & (pred_present_mask)].shape[0]

# FP: 予測にはあるが正解にはない
FP = merged_df[(~gt_present_mask) & (pred_present_mask)].shape[0]

# FN: 正解にはあるが予測にはない
FN = merged_df[(gt_present_mask) & (~pred_present_mask)].shape[0]

print("\n--- 開示情報の検出精度 ---")
print(f"True Positives (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")


# =========================
# 7) 適合率（Precision）と再現率（Recall）を計算
# =========================

# 適合率：予測したもののうち、正しかった割合
precision = TP / (TP + FP) if (TP + FP) > 0 else 0

# 再現率：正解のうち、予測できた割合
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

print(f"Precision (適合率): {precision:.4f}")
print(f"Recall (再現率): {recall:.4f}")


# =========================
# 8) TPの中で「発行数」まで一致しているか（値の抽出精度）
# =========================

# TPだけを取り出す（正解にも予測にも値がある行）
tp_matches = merged_df[(gt_present_mask) & (pred_present_mask)].copy()

# 発行数が完全一致している件数をカウント
value_matches = (tp_matches[value_col + '_gt'] == tp_matches[value_col + '_pred']).sum()

# 比較できたTPの総数
total_tp_with_values = tp_matches.shape[0]

# TP内での発行数一致率
value_accuracy = value_matches / total_tp_with_values if total_tp_with_values > 0 else 0

print("\n--- 発行数の抽出精度 (TP内) ---")
print(f"True Positives (TP) のうち、発行数が一致した件数: {value_matches}")
print(f"発行数を比較できた True Positives (TP) 総数: {total_tp_with_values}")
print(f"Value Accuracy (発行数の一致率): {value_accuracy:.4f}")


# =========================
# 9) 詳細表示（FN / FP / 発行数不一致TP）
# =========================

# FN（正解にはあるが予測できなかった開示）を表示
if FN > 0:
    print("\n--- False Negatives (FN) の詳細 ---")
    fn_rows = merged_df[(gt_present_mask) & (~pred_present_mask)]
    display(fn_rows[key_cols + [value_col + '_gt']])

# FP（予測したが正解には存在しない開示）を表示
if FP > 0:
    print("\n--- False Positives (FP) の詳細 ---")
    fp_rows = merged_df[(~gt_present_mask) & (pred_present_mask)]
    display(fp_rows[key_cols + [value_col + '_pred']])

# TPだが発行数が一致しないケースを表示
if value_matches < total_tp_with_values:
    print("\n--- True Positives (TP) だが発行数が不一致の詳細 ---")
    value_mismatch_rows = tp_matches[tp_matches[value_col + '_gt'] != tp_matches[value_col + '_pred']]
    display(value_mismatch_rows[key_cols + [value_col + '_gt', value_col + '_pred']])


In [ ]:
print(f"新株予約権の新規発行がある開示情報の件数: {disclosure_count_final}")
print(f"新株予約権の新規発行数の合計: {int(newshare_total_final)}")